# 📄 Gerando Dataset de Instruções para Fine‑Tuning a partir de Documentos

Este tutorial ensina como transformar conhecimento bruto — contido em manuais, guias ou artigos (PDF/Texto) — em um conjunto de dados no formato **instruction‑input‑output**, pronto para ser usado em fine‑tuning de modelos de linguagem (como o que fizemos com LoRA).

**Por que isso é importante?**
- Modelos pré‑treinados possuem grande capacidade, mas frequentemente carecem de conhecimento específico de domínio.
- Ao criar pares pergunta‑resposta baseados em documentos reais, podemos ensinar o modelo a responder corretamente sobre aquele domínio.
- O formato JSONL utilizado é o padrão para *instruction tuning*, amplamente adotado em projetos como Alpaca, Dolly, etc.

**O que você aprenderá:**
- Extrair texto de arquivos PDF.
- Dividir o texto em trechos (chunks) adequados para processamento.
- Usar tanto um modelo **seq2seq** (encoder‑decoder) quanto um modelo **causal** (autoregressivo) para gerar automaticamente triplas `instruction`, `input` (opcional) e `output`.
- Estruturar e salvar os dados em JSONL.
- Comparar a qualidade dos dados gerados com uma extração manual simples.

## 📚 1. Fundamentos da Geração de Dados Instrucionais

O objetivo é, dado um trecho de texto $D$ (ex.: um parágrafo de manual), produzir uma tripla $(I, X, O)$ onde:
- $I$: instrução (pergunta ou comando)
- $X$: entrada adicional (opcional, pode ser vazia)
- $O$: saída desejada, baseada no conteúdo de $D$

Formalmente, queremos modelar uma distribuição condicional:

$$P(I, X, O \mid D)$$

Utilizamos um LLM para amostrar dessas distribuições, fornecendo um *prompt* que instrui o modelo a gerar o JSON desejado a partir do contexto.

**Por que isso funciona?**  
Modelos de linguagem modernos, quando condicionados com prompts adequados, conseguem realizar *in‑context learning* – eles entendem a tarefa descrita e produzem texto estruturado. A qualidade depende fortemente:
- Da capacidade do modelo (ex.: 7B+ parâmetros).
- Da clareza do prompt.
- Da temperatura (controla a criatividade).

Neste notebook demonstraremos dois paradigmas:
- **Seq2Seq** (ex.: FLAN‑T5): recebe o texto inteiro e gera a saída condicionada ao *encoder*.
- **Causal** (ex.: GPT‑2): gera a continuação de um prompt, simulando a tarefa como um "completamento" de texto.

> **Nota didática:** Utilizaremos modelos pequenos para demonstrar o fluxo. Em aplicações reais, recomenda‑se modelos maiores (LLaMA 3, GPT‑4, etc.) para melhores resultados.

## 📦 2. Requisitos

Instale as bibliotecas necessárias (descomente a linha se ainda não as tiver):

In [1]:
!pip install -r requirements.txt

In [10]:
!pip install protobuf

In [2]:
import pdfplumber
import json
import torch
from transformers import pipeline, logging, AutoTokenizer, AutoModelForSeq2SeqLM
from tqdm import tqdm
from PyPDF2 import PdfReader
# Desativa avisos (apenas erros críticos serão mostrados)
logging.set_verbosity_error()

/home/joaovitor/Documents/Faculdade/1°Semestre/Tópicos avançados em Inteligência Artificial A/Second Avaliation/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
def extract_text_from_pdf(pdf_path):
    """Extrai texto de um arquivo PDF."""
    reader = PdfReader(pdf_path)
    text = ""
    for page in reader.pages:
        page_text = page.extract_text()
        if page_text:
            text += page_text + "\n"
    return text

# Substitua pelo caminho do seu PDF
pdf_path = "Dataset.pdf"
full_text = extract_text_from_pdf(pdf_path)
print(f"Total de caracteres extraídos: {len(full_text)}")
print("\n--- INÍCIO DO TEXTO ---\n")
print(full_text[:500])

Total de caracteres extraídos: 278894

--- INÍCIO DO TEXTO ---

RELATÓRIO
DE GESTÃO
DEPARTAMENTO
NACIONAL2021

RELATÓRIO
DE GESTÃO
DEPARTAMENTO
NACIONAL2021
CONFEDERAÇÃO NACIONAL DA INDÚSTRIA – CNI
Robson Braga de Andrade
Presidente
Gabinete da Presidência 
Teodomiro Braga da Silva 
Chefe do Gabinete - Diretor
Diretoria de Educação e Tecnologia – DIRET
Rafael Esmeraldo Lucchesi Ramacciotti
Diretor de Educação e Tecnologia
Serviço Social da Indústria – SESI
Eduardo Eugenio Gouvêa Vieira
Presidente do Conselho Nacional
SESI – Departamento Nacional
Robson Braga


In [4]:
def chunk_text(text, max_chunk_chars=500):
    """Divide o texto em blocos de no máximo max_chunk_chars caracteres."""
    words = text.split()
    chunks = []
    current_chunk = ""
    for word in words:
        if len(current_chunk) + len(word) + 1 <= max_chunk_chars:
            current_chunk += (" " if current_chunk else "") + word
        else:
            chunks.append(current_chunk)
            current_chunk = word
    if current_chunk:
        chunks.append(current_chunk)
    return chunks

chunks = chunk_text(full_text, max_chunk_chars=200)  # pequeno para demonstração
print(f"Número de chunks: {len(chunks)}")
print(f"Exemplo de chunk (primeiro):\n{chunks[0][:300]}...")

Número de chunks: 1369
Exemplo de chunk (primeiro):
RELATÓRIO DE GESTÃO DEPARTAMENTO NACIONAL2021 RELATÓRIO DE GESTÃO DEPARTAMENTO NACIONAL2021 CONFEDERAÇÃO NACIONAL DA INDÚSTRIA – CNI Robson Braga de Andrade Presidente Gabinete da Presidência...


In [5]:
def split_text(text, max_chunk_length=500):
    """
    Divide o texto em chunks com base em quebras de linha.
    Cada chunk terá no máximo max_chunk_length caracteres.
    """
    paragraphs = text.split("\n")
    chunks = []
    current_chunk = ""

    for para in paragraphs:
        if len(current_chunk) + len(para) < max_chunk_length:
            current_chunk += para + "\n"
        else:
            if current_chunk:
                chunks.append(current_chunk.strip())
            current_chunk = para + "\n"

    if current_chunk:
        chunks.append(current_chunk.strip())

    return chunks


In [6]:
def generate_instruction_response(chunk, hf_pipeline):
    """
    Dado um chunk de texto, usa o pipeline de texto para gerar uma pergunta curta
    e uma resposta direta (ambas com menos de 10 palavras).
    """
    prompt = f"""
You are a data generator for fine-tuning or training a language model.

Given the content below, create:
1. A short and objective question (instruction), under 10 words, based only on the content.
2. A direct and concise answer (response), under 10 words, based only on the content.

Do not add extra explanations. Use the following format:

INSTRUCTION: <short question>
RESPONSE: <short answer>

Content:
\"\"\"
{chunk}
\"\"\"
"""

    messages = [{"role": "user", "content": prompt}]

    try:
        outputs = hf_pipeline(
            messages,
            max_new_tokens=150,
            max_length=None,  # evita warning
            return_full_text=False
        )
        content = outputs[0]["generated_text"]

        # Extrai os campos esperados
        instr_part = content.split("INSTRUCTION:")[1].split("RESPONSE:")[0].strip()
        answer = content.split("RESPONSE:")[1].strip()
        return instr_part, answer
    except Exception as e:
        # Em caso de erro, retorna None, None (o par será ignorado)
        return None, None


In [7]:
def save_to_jsonl(pairs, output_file="dataset.jsonl"):
    """
    pairs: lista de tuplas (instrução, resposta)
    output_file: nome do arquivo de saída
    """
    with open(output_file, "w", encoding="utf-8") as f:
        for instruction, answer in pairs:
            if instruction and answer:
                example = {
                    "Instruction": instruction,
                    "Output": answer
                }
                f.write(json.dumps(example, ensure_ascii=False) + "\n")


In [8]:
def generate_dataset(file_path, model_id="google/mt5-base",
                    output_file="dataset.jsonl", max_chunks=None):
    """
    file_path: caminho para o arquivo .pdf ou .txt
    model_id: identificador do modelo no Hugging Face Hub
    output_file: nome do arquivo de saída (JSONL)
    max_chunks: se especificado, limita a quantidade de chunks processados
    """
    print(f"🔄 Carregando modelo: {model_id} ...")

    # Pipeline para geração de texto
    hf_pipeline = pipeline(
        "text2text-generation",
        model=model_id,
        device_map="auto",        # usa GPU se disponível
        torch_dtype=torch.bfloat16 # reduz consumo de memória
    )

    print("📄 Extraindo texto do arquivo...")
    text = extract_text_from_file(file_path)

    print("✂️  Dividindo em chunks...")
    chunks = split_text(text)

    if max_chunks:
        chunks = chunks[:max_chunks]

    print(f"🧠 Gerando pares (instrução + resposta) para {len(chunks)} chunks...")
    pairs = []

    for chunk in tqdm(chunks, desc="Processando chunks"):
        if len(chunk.strip()) < 10:  # ignora chunks muito curtos
            continue
        instruction, answer = generate_instruction_response(chunk, hf_pipeline)
        if instruction and answer:
            pairs.append((instruction, answer))

    save_to_jsonl(pairs, output_file)
    print(f"\n✅ Dataset salvo em: {output_file} ({len(pairs)} exemplos gerados)")


In [13]:
# Exemplo de uso com um arquivo .pdf
caminho_arquivo = "Dataset.pdf"   # Altere para o seu arquivo
# caminho_arquivo = "meu_texto.txt"      # Ou use um .txt

generate_dataset(
    file_path=caminho_arquivo,
    model_id="allenai/unifiedqa-t5-3b",
    output_file="dataset_gerado_500.jsonl",
    max_chunks=10   # Número maximo de perguntas e respostas
)

🔄 Carregando modelo: allenai/unifiedqa-t5-3b ...
📄 Extraindo texto do arquivo...


NameError: name 'extract_text_from_file' is not defined